# IEEE-CIS Fraud Detection: Feature Engineering

**Goal:** Transform raw data into model-ready features based on EDA insights

**Key EDA Insights to Leverage:**
1. Email match is a strong fraud signal (9.6% vs 2.2%)
2. New cards (low D1) have higher fraud rates
3. Time-based patterns exist in hour/day
4. Mobile devices show higher fraud rates
5. V features cluster by missing patterns
6. Transaction amount patterns differ by fraud status

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

from utils import disp_columns

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

## 1. Load Data

In [11]:
# Load training data
train_txn = pd.read_csv(DATA_DIR / 'train_transaction.csv')
train_id = pd.read_csv(DATA_DIR / 'train_identity.csv')

# Load test data
test_txn = pd.read_csv(DATA_DIR / 'test_transaction.csv')
test_id = pd.read_csv(DATA_DIR / 'test_identity.csv')

print(f"Train transactions: {train_txn.shape}")
print(f"Test transactions: {test_txn.shape}")

Train transactions: (590540, 394)
Test transactions: (506691, 393)


In [12]:
# Merge transaction and identity data
train = pd.merge(train_txn, train_id, on='TransactionID', how='left')
test = pd.merge(test_txn, test_id, on='TransactionID', how='left')

# Store target and IDs separately
y_train = train['isFraud'].copy()
train_ids = train['TransactionID'].copy()
test_ids = test['TransactionID'].copy()

print(f"Train shape after merge: {train.shape}")
print(f"Test shape after merge: {test.shape}")

Train shape after merge: (590540, 434)
Test shape after merge: (506691, 433)


In [13]:

print("Columns in train but not in test: ", len(set(train.columns) - set(test.columns)))
print("Columns in test but not in train: ", len(set(test.columns) - set(train.columns)))

disp_columns(train)
disp_columns(test)

Columns in train but not in test:  39
Columns in test but not in train:  38
┌────────────────────────────────────────────────┐
│ - TransactionID: int64                         │
│ - isFraud: int64                               │
│ - TransactionDT: int64                         │
│ - TransactionAmt: float64                      │
│ - ProductCD: object                            │
│ - card<N>                                      │
│   - N = 1: int64                               │
│   - N = 2-3, 5: float64                        │
│   - N = 4, 6: object                           │
│ - addr<1-2>: float64                           │
│ - dist<1-2>: float64                           │
│ - P_emaildomain: object                        │
│ - R_emaildomain: object                        │
│ - C<1-14>: float64                             │
│ - D<1-15>: float64                             │
│ - M<1-9>: object                               │
│ - V<1-339>: float64                            │
│ - id

## 2. Build Pipeline

### Column name normalization

The test data has columns `id-01`, `id-02`, etc. (with "-") whereas the train data has columns `id_01`, `id_02`, etc. (with "_"). We add an initial step (`ColumnNormalizer`) to the pipeline to normalize the column names.

### Encoding Strategy

Based on EDA analysis of feature distributions:

| Column(s) | Encoding | Reason |
|-----------|----------|--------|
| `card1`, `addr1` | Frequency | High cardinality, used to build composite customer UID |
| `id_01`, `id_03`–`id_06`, `id_09`–`id_11`, `id_13`, `id_14`, `id_17`–`id_22`, `id_24`–`id_26` | Frequency | Numeric dtype but categorical behavior (discrete peaks in distribution) |
| `id_32` | Label | Low cardinality (4 values: 0, 16, 24, 32 — likely screen color depth) |
| `id_02`, `id_07`, `id_08` | None (keep numeric) | True continuous distributions |
| `card4`, `card6`, `ProductCD`, `M1`–`M9`, `DeviceType`, etc. | Label | Already string dtype, picked up by `CategoricalEncoder` |

We will create two pipelines: one for lightGBM and tree-based methods, using label-encoding for categorical fields; and one using one-hot encoding for these fields which will be more suitable for regression techniques. 

### New features

#### Timestamp features

In order to utilize patterns in the `TransactionDT` timestamps we create new features with the `TimeFeatures` transformer to capture the hour-of-day (`hod_sin`, `hod_cos`) and day-of-week (`dow_sin`, `dow_cos`).

#### Email features

We also create new features based on the email addresses in `P_emaildomain` and `R_emaildomain`:
- `email_match`: whether the `P_emaildomain` and `R_emaildomain` are the same
- `P_email_is_free` and `R_email_is_free`: whether the domain is a free email domain
- `P_email_missing` and `R_email_missing`: whether the domain is missing

#### CardFeatures

- `is_new_card`: whether the card is new, `D1 <= 7`
- `has_identity`: whether the card is present in `identity`
- `is_mobile`: whether `DeviceType == 'mobile'`

#### Amount features

- `TransactionAmt_log`
- `TransactionAmt_decimal`
- `TransactionAmt_is_round`

#### Missing Indicators

- Boolean `D8_missing` etc. for the `D\d+` fields
- A count of how many `V\d+` fields are missing

In [14]:
from sklearn.pipeline import Pipeline

from transformers import (
    ColumnNormalizer,
    TimeFeatures,
    EmailFeatures,
    CardFeatures,
    AmountFeatures,
    AggregationFeatures,
    FrequencyEncoder,
    MissingIndicators,
    AsCategory,
    CategoricalEncoder,
    OneHotEncoder,
)

# Columns identified in EDA as needing special encoding:
# - High-cardinality: frequency encode (card1, addr1, most numeric id cols)
# - Low-cardinality categorical: convert to string for label encoding (id_32)
# - True numeric: leave as-is (id_02, id_07, id_08)

ID_FREQ_COLS = [
    'id_01', 'id_03', 'id_04', 'id_05', 'id_06', 'id_09', 'id_10', 'id_11',
    'id_13', 'id_14', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22',
    'id_24', 'id_25', 'id_26',
]

# Shared preprocessing steps (before categorical encoding)
SHARED_STEPS = [
    ('normalize_columns', ColumnNormalizer()),
    ('time', TimeFeatures()),
    ('email', EmailFeatures()),
    ('card', CardFeatures()),
    ('amount', AmountFeatures()),
    ('aggregation', AggregationFeatures(uid_cols=['card1', 'addr1'])),
    ('frequency', FrequencyEncoder(cols=['card1', 'addr1'] + ID_FREQ_COLS)),
    ('missing', MissingIndicators()),
    ('as_category', AsCategory(cols=['id_32'])),
]

# Pipeline for tree-based models (LightGBM, XGBoost)
# Uses label encoding - trees can handle arbitrary numeric splits
tree_pipeline = Pipeline(SHARED_STEPS + [
    ('categorical', CategoricalEncoder()),
])

# Pipeline for linear models (Logistic Regression, SVM)
# Uses one-hot encoding - linear models need proper categorical representation
linear_pipeline = Pipeline(SHARED_STEPS + [
    ('categorical', OneHotEncoder(max_categories=50)),
])

print("Tree pipeline (for LightGBM):")
for name, step in tree_pipeline.steps:
    print(f"  - {name}: {step.__class__.__name__}")

print("\nLinear pipeline (for Logistic Regression):")
for name, step in linear_pipeline.steps:
    print(f"  - {name}: {step.__class__.__name__}")

Tree pipeline (for LightGBM):
  - normalize_columns: ColumnNormalizer
  - time: TimeFeatures
  - email: EmailFeatures
  - card: CardFeatures
  - amount: AmountFeatures
  - aggregation: AggregationFeatures
  - frequency: FrequencyEncoder
  - missing: MissingIndicators
  - as_category: AsCategory
  - categorical: CategoricalEncoder

Linear pipeline (for Logistic Regression):
  - normalize_columns: ColumnNormalizer
  - time: TimeFeatures
  - email: EmailFeatures
  - card: CardFeatures
  - amount: AmountFeatures
  - aggregation: AggregationFeatures
  - frequency: FrequencyEncoder
  - missing: MissingIndicators
  - as_category: AsCategory
  - categorical: OneHotEncoder


## 3. Fit and Transform

In [15]:
# Fit and transform with TREE pipeline (for LightGBM)
train_tree = tree_pipeline.fit_transform(train)
test_tree = tree_pipeline.transform(test)

print(f"Tree pipeline - Train shape: {train_tree.shape}")
print(f"Tree pipeline - Test shape: {test_tree.shape}")

# Fit and transform with LINEAR pipeline (for Logistic Regression)
linear_pipeline_fitted = Pipeline(SHARED_STEPS + [
    ('categorical', OneHotEncoder(max_categories=50)),
])
train_linear = linear_pipeline_fitted.fit_transform(train)
test_linear = linear_pipeline_fitted.transform(test)

print(f"\nLinear pipeline - Train shape: {train_linear.shape}")
print(f"Linear pipeline - Test shape: {test_linear.shape}")
print(f"\nOne-hot encoding added {train_linear.shape[1] - train_tree.shape[1]} features")

Tree pipeline - Train shape: (590540, 457)
Tree pipeline - Test shape: (506691, 456)

Linear pipeline - Train shape: (590540, 522)
Linear pipeline - Test shape: (506691, 521)

One-hot encoding added 65 features


In [16]:
# Select features (drop ID, target, raw time)
drop_cols = ['TransactionID', 'isFraud', 'TransactionDT']

def select_features(train_df, test_df, drop_cols):
    """Select common features between train and test."""
    common_cols = set(train_df.columns) & set(test_df.columns)
    feature_cols = [c for c in train_df.columns if c in common_cols and c not in drop_cols]
    return train_df[feature_cols], test_df[feature_cols], feature_cols

# Tree pipeline features
X_train_tree, X_test_tree, tree_feature_cols = select_features(train_tree, test_tree, drop_cols)
print(f"Tree pipeline - Features: {len(tree_feature_cols)}")

# Linear pipeline features  
X_train_linear, X_test_linear, linear_feature_cols = select_features(train_linear, test_linear, drop_cols)
print(f"Linear pipeline - Features: {len(linear_feature_cols)}")

Tree pipeline - Features: 454
Linear pipeline - Features: 519


## 4. Save Processed Data and Pipeline

## 2. Build Pipeline

### Column name normalization

The test data has columns `id-01`, `id-02`, etc. (with "-") whereas the train data has columns `id_01`, `id_02`, etc. (with "_"). We add an initial step (`ColumnNormalizer`) to the pipeline to normalize the column names.

### Encoding Strategy

Based on EDA analysis of feature distributions:

| Column(s) | Encoding | Reason |
|-----------|----------|--------|
| `card1`, `addr1` | Frequency | High cardinality, used to build composite customer UID |
| `id_01`, `id_03`–`id_06`, `id_09`–`id_11`, `id_13`, `id_14`, `id_17`–`id_22`, `id_24`–`id_26` | Frequency | Numeric dtype but categorical behavior (discrete peaks in distribution) |
| `id_32` | Label | Low cardinality (4 values: 0, 16, 24, 32 — likely screen color depth) |
| `id_02`, `id_07`, `id_08` | None (keep numeric) | True continuous distributions |
| `card4`, `card6`, `ProductCD`, `M1`–`M9`, `DeviceType`, etc. | Label | Already string dtype, picked up by `CategoricalEncoder` |

We will create two pipelines: one for lightGBM and tree-based methods, using label-encoding for categorical fields; and one using one-hot encoding for these fields which will be more suitable for regression techniques. 

### New features

#### Timestamp features
- `hod_sin`, `hod_cos`: Cyclical hour-of-day encoding
- `dow_sin`, `dow_cos`: Cyclical day-of-week encoding

#### Email features (EDA Insight 6: email match = 9.7% fraud vs 2.2%)
- `email_match`: P_emaildomain == R_emaildomain
- `P_email_is_free`, `R_email_is_free`: Free email provider flags
- `P_email_missing`, `R_email_missing`: Missing domain flags
- **NEW** `P_email_high_risk`, `R_email_high_risk`: Outlook/hotmail flags (9.5% fraud rate)

#### Card features (EDA Insight 8: new cards have elevated fraud)
- `is_new_card`: D1 <= 7 days
- `has_identity`: DeviceType is not null
- `is_mobile`: DeviceType == 'mobile'
- **NEW** `card_age_bucket`: Granular D1 buckets (0d, 1-7d, 8-30d, 31-90d, 91-365d, 365d+)

#### Device features (EDA Insight 6: mobile 10.2% vs desktop 6.5%)
- **NEW** `is_windows`, `is_macos`, `is_ios`, `is_android`, `is_linux`: OS flags
- **NEW** `is_chrome`, `is_safari`, `is_firefox`, `is_edge`: Browser flags

#### Interaction features (EDA Insights 4-5: card network × product patterns)
- **NEW** `card4_ProductCD`, `card6_ProductCD`: Card network × product
- **NEW** `DeviceType_ProductCD`: Device × product
- **NEW** `card4_DeviceType`: Card network × device

#### Amount features (EDA Insight 2: non-monotonic fraud rate by amount)
- `TransactionAmt_log`: Log-transformed amount
- `TransactionAmt_is_round`: Round dollar amount flag
- **NEW** `amt_bucket`: Amount buckets ($0-25, $25-50, ..., $1000+)
- **NEW** `cents`, `cents_00`, `cents_99`, `cents_95`: Cents pattern features

#### Missing Indicators (EDA Insights 7-8: V/D missing patterns)
- `D8_missing`, `D7_missing`, `D2_missing`: D feature missing flags
- `v_missing_count`: Count of missing V features
- **NEW** `V1_11_missing`, `V12_34_missing`, ...: V block missing indicators

from sklearn.pipeline import Pipeline

from transformers import (
    ColumnNormalizer,
    TimeFeatures,
    EmailFeatures,
    CardFeatures,
    DeviceFeatures,      # NEW: OS/browser parsing
    InteractionFeatures, # NEW: card4 × ProductCD etc.
    AmountFeatures,
    AggregationFeatures,
    FrequencyEncoder,
    MissingIndicators,
    VFeatureBlocks,      # NEW: V feature block missing indicators
    AsCategory,
    CategoricalEncoder,
    OneHotEncoder,
)

# Columns identified in EDA as needing special encoding:
# - High-cardinality: frequency encode (card1, addr1, most numeric id cols)
# - Low-cardinality categorical: convert to string for label encoding (id_32)
# - True numeric: leave as-is (id_02, id_07, id_08)

ID_FREQ_COLS = [
    'id_01', 'id_03', 'id_04', 'id_05', 'id_06', 'id_09', 'id_10', 'id_11',
    'id_13', 'id_14', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22',
    'id_24', 'id_25', 'id_26',
]

# Shared preprocessing steps (before categorical encoding)
# Order matters: some steps create columns used by later steps
SHARED_STEPS = [
    ('normalize_columns', ColumnNormalizer()),
    ('time', TimeFeatures()),
    ('email', EmailFeatures()),           # Now includes high-risk domain flags
    ('card', CardFeatures()),             # Now includes card_age_bucket
    ('device', DeviceFeatures()),         # NEW: is_windows, is_ios, etc.
    ('interactions', InteractionFeatures()),  # NEW: card4_ProductCD, etc.
    ('amount', AmountFeatures()),         # Now includes amt_bucket, cents features
    ('aggregation', AggregationFeatures(uid_cols=['card1', 'addr1'])),
    ('frequency', FrequencyEncoder(cols=['card1', 'addr1'] + ID_FREQ_COLS)),
    ('missing', MissingIndicators()),
    ('v_blocks', VFeatureBlocks()),       # NEW: V1_11_missing, V12_34_missing, etc.
    ('as_category', AsCategory(cols=['id_32'])),
]

# Pipeline for tree-based models (LightGBM, XGBoost)
# Uses label encoding - trees can handle arbitrary numeric splits
tree_pipeline = Pipeline(SHARED_STEPS + [
    ('categorical', CategoricalEncoder()),
])

# Pipeline for linear models (Logistic Regression, SVM)
# Uses one-hot encoding - linear models need proper categorical representation
linear_pipeline = Pipeline(SHARED_STEPS + [
    ('categorical', OneHotEncoder(max_categories=50)),
])

print("Tree pipeline (for LightGBM):")
for name, step in tree_pipeline.steps:
    print(f"  - {name}: {step.__class__.__name__}")

print("\nLinear pipeline (for Logistic Regression):")
for name, step in linear_pipeline.steps:
    print(f"  - {name}: {step.__class__.__name__}")

In [ ]:
new_features = [
    # Time features
    'hod_sin', 'hod_cos', 'dow_sin', 'dow_cos',
    
    # Email features (including NEW high-risk flags)
    'email_match', 'P_email_is_free', 'P_email_high_risk', 'R_email_high_risk',
    
    # Card features (including NEW card_age_bucket)
    'is_new_card', 'has_identity', 'is_mobile', 'card_age_bucket',
    
    # NEW Device features
    'is_windows', 'is_macos', 'is_ios', 'is_android',
    'is_chrome', 'is_safari', 'is_firefox',
    
    # Amount features (including NEW bucket and cents)
    'TransactionAmt_log', 'amt_bucket', 
    'cents', 'cents_00', 'cents_99',
    
    # Missing indicators (including NEW V block indicators)
    'v_missing_count', 'V1_11_missing', 'V12_34_missing', 'V95_137_missing',
    
    # UID-based aggregation features
    'card1_addr1_amt_mean', 'card1_addr1_amt_std', 
    'card1_addr1_txn_count', 'card1_addr1_amt_zscore',
    
    # Frequency encoding features
    'card1_freq', 'addr1_freq',
]

print("New feature statistics (tree pipeline):")
X_train_tree[new_features].describe().round(3)